# Stage 3 — DPO Preference Alignment
Loads the Stage-2 SFT adapter and aligns it on `data/preference_dataset.jsonl`
((prompt, chosen, rejected) triples).

In [1]:
# reconnect Drive and set the project path
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT = '/content/drive/MyDrive/Domain-SupportSpecialist'

# Sanity check: fail loudly and clearly if the project folder isn't where expected ---
REQUIRED_FILES = [
    'data/preference_dataset.jsonl',
    'adapters/stage2_sft/adapter_config.json',
    'reports/base_model_evaluation.md',
    'reports/sft_model_comparison.md',
]
missing = [p for p in REQUIRED_FILES if not os.path.exists(f'{PROJECT}/{p}')]
if missing:
    raise FileNotFoundError(
        "Missing from PROJECT: " + ", ".join(missing) +
        "\n\nIf 'data/preference_dataset.jsonl' is missing: confirm 'My Drive/Domain-SupportSpecialist/' is at the TOP level of Drive (not nested).\nIf 'adapters/stage2_sft/...' is missing: run notebooks/instruction_finetuning.ipynb (Stage 2) all the way through first.\nIf either report .md is missing: those come from Stage 1 and Stage 2 respectively -- make sure both finished (including their final report-writing cells) before this one."
    )

for sub in ['data', 'notebooks', 'reports', 'src',
            'adapters/stage1_non_instruction', 'adapters/stage2_sft', 'adapters/stage3_dpo']:
    os.makedirs(f'{PROJECT}/{sub}', exist_ok=True)

print('Project root OK:', PROJECT)

Mounted at /content/drive
Project root OK: /content/drive/MyDrive/Domain-SupportSpecialist


In [2]:
# fail fast if there's no GPU, with clear instructions
import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected in this runtime.\n"
        "Fix: Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU -> Save, "
        "then Runtime -> Restart session, then run all cells again from the top.\n"
        "If T4 GPU won't attach at all / stays on 'None', you may be out of free GPU quota "
        "for today on this Google account -- try again later, or use a different account."
    )
print("GPU OK:", torch.cuda.get_device_name(0))

GPU OK: Tesla T4


In [3]:
%%capture
# Install Unsloth. NOTE: do not pin an old trl version here -- unsloth_zoo needs a
# modern trl, and forcing an old one causes both import errors and a
# 'Trainer.__init__() got an unexpected keyword argument tokenizer' crash later
# (modern trl/transformers renamed tokenizer= to processing_class=).
!pip install --no-deps unsloth
!pip install unsloth_zoo
!pip install --no-deps peft accelerate bitsandbytes xformers

In [4]:
# load the SFT model and re-attach a fresh LoRA for DPO
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = f"{PROJECT}/adapters/stage2_sft",
    max_seq_length = 2048, dtype = None, load_in_4bit = True,
)
model = FastLanguageModel.get_peft_model(
    model, r = 16, lora_alpha = 16, lora_dropout = 0,
    target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Unsloth 2026.7.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.
Unsloth: Already have LoRA adapters! We shall skip this step.


In [5]:
# format the preference dataset for DPOTrainer
import json
from datasets import Dataset

pref_rows = [json.loads(l) for l in open(f"{PROJECT}/data/preference_dataset.jsonl")]

def fmt(p):
    return tokenizer.apply_chat_template(
        [{"role":"user","content":p}], tokenize=False, add_generation_prompt=True)

dpo_ds = Dataset.from_dict({
    "prompt":   [fmt(r["prompt"]) for r in pref_rows],
    "chosen":   [r["chosen"] for r in pref_rows],
    "rejected": [r["rejected"] for r in pref_rows],
})
print(len(dpo_ds), "preference pairs")

100 preference pairs


In [6]:
# run DPO
from trl import DPOTrainer, DPOConfig

dpo_trainer = DPOTrainer(
    model = model, ref_model = None, processing_class = tokenizer, train_dataset = dpo_ds,
    args = DPOConfig(
        per_device_train_batch_size = 2, gradient_accumulation_steps = 4,
        warmup_steps = 5, num_train_epochs = 2, learning_rate = 5e-6,
        logging_steps = 5, output_dir = "stage3_out", optim = "adamw_8bit",
        beta = 0.1, max_length = 1024, max_prompt_length = 512,
        save_strategy = "no",
    ),
)
dpo_trainer.train()

import os
os.makedirs(f"{PROJECT}/adapters/stage3_dpo", exist_ok=True)
model.save_pretrained(f"{PROJECT}/adapters/stage3_dpo")
tokenizer.save_pretrained(f"{PROJECT}/adapters/stage3_dpo")

Extracting prompt in train dataset (num_proc=6):   0%|          | 0/100 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=6):   0%|          | 0/100 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=6):   0%|          | 0/100 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 100 | Num Epochs = 2 | Total steps = 26
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected
5,0.027140,7.974957,-0.345100,1.000000,8.320057,-102.465576,-44.444637,-2.153600,-0.337748
10,0.016368,7.121417,-0.476804,1.000000,7.598220,-108.764870,-49.324379,-1.969431,-0.317582
15,0.030937,8.780620,-0.559783,1.000000,9.340404,-115.221474,-48.430531,-2.118270,-0.378282
20,0.024462,7.618055,-0.616924,1.000000,8.234980,-109.767288,-47.005024,-2.021189,-0.377271
25,0.008460,8.730494,-0.555098,1.000000,9.285591,-103.027267,-48.965279,-2.107288,-0.339686


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/Domain-SupportSpecialist/adapters/stage3_dpo/tokenizer_config.json.


('/content/drive/MyDrive/Domain-SupportSpecialist/adapters/stage3_dpo/tokenizer_config.json',
 '/content/drive/MyDrive/Domain-SupportSpecialist/adapters/stage3_dpo/chat_template.jinja',
 '/content/drive/MyDrive/Domain-SupportSpecialist/adapters/stage3_dpo/tokenizer.json')

In [7]:
# the same 10 questions again
QUESTIONS = [
    "How can I cancel my order after it has been placed?",
    "My package says delivered but I never received it, what do I do?",
    "How long does a refund take to appear on my card?",
    "Can I change the delivery address after checkout?",
    "The item I received is damaged, what are my options?",
    "How do I track my order status?",
    "I was charged twice for one order, how do I fix this?",
    "Can I get a replacement instead of a refund?",
    "What happens if I miss the delivery attempt?",
    "How do I apply a discount code after placing an order?",
]

def ask(prompt):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True,
                add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = model.generate(input_ids=inputs, max_new_tokens=200, temperature=0.7)
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)

FastLanguageModel.for_inference(model)
dpo_answers = {q: ask(q) for q in QUESTIONS}
for q, a in dpo_answers.items():
    print("Q:", q, "\nA:", a, "\n---")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_

Q: How can I cancel my order after it has been placed? 
A: I've got all the information you need to cancel your order! Rest assured, we understand that it may be frustrating to see an order on hold or unable to be checked in our system. To assist you with this process, please follow these steps:

1. Log into Your Account: Begin by logging into your account portal at support.example.com using your credentials.
2. Access Your Order History: Once logged in, navigate to the "Order History" section.
3. Locate and Select the Relevant Order: Look for any orders associated with the #SC-26784 number and select them.
4. Initiate Cancellation: Within the selected order details, locate the option labeled "Order History" and click on it.
5. Confirm Your Decision: The system will prompt you to confirm whether you want to cancel the order. Please provide your feedback.

If you encounter any difficulties during the cancellation process or have further questions, our dedicated customer service team is 

In [8]:
# recover base_answers and sft_answers from the earlier markdown reports,
# then write the final three-way evaluation
def parse_report(path, col_index):
    with open(path, encoding='utf-8') as f:
        text = f.read()

    # anchor-based extraction, not a naive per-line parser -- several answers
    # span multiple physical lines (numbered steps), and a plain "keep lines
    # starting with |" filter silently truncates those answers to just their
    # first line instead of capturing the full text
    answers = {}
    anchors = []
    for q in QUESTIONS:
        marker = f"| {q} |"
        idx = text.find(marker)
        if idx != -1:
            anchors.append((idx, q))
    anchors.sort()

    for i, (idx, q) in enumerate(anchors):
        end = anchors[i + 1][0] if i + 1 < len(anchors) else len(text)
        block = text[idx:end].strip()
        parts = [p.strip() for p in block.split('|')]
        parts = [p for p in parts if p != ''] if parts and parts[0] == '' else parts
        if len(parts) > col_index:
            answers[q] = parts[col_index]
    return answers

base_answers = parse_report(f'{PROJECT}/reports/base_model_evaluation.md', 1)
sft_answers  = parse_report(f'{PROJECT}/reports/sft_model_comparison.md', 2)

rows_md = "\n".join(
    f"| {q} | {base_answers.get(q,'_(see report)_')} | {sft_answers.get(q,'_(see report)_')} | {dpo_answers[q].strip()} | _(fill in)_ | _(fill in)_ |"
    for q in QUESTIONS
)
report = f"""# Final Evaluation -- Base vs SFT vs DPO

| Question | Base | SFT | DPO | Best Answer | Reason |
|---|---|---|---|---|---|
{rows_md}
"""
with open(f'{PROJECT}/reports/final_evaluation.md', 'w', encoding='utf-8') as f:
    f.write(report)


In [9]:
# copy this executed notebook into notebooks/ on Drive (download + push manually afterward)
import shutil, glob, os

candidates = glob.glob('/content/drive/MyDrive/Colab Notebooks/*.ipynb')
if candidates:
    src = max(candidates, key=os.path.getmtime)
    shutil.copy(src, f'{PROJECT}/notebooks/dpo_alignment.ipynb')
